# Analyser ses données trail (3/3) — Lire sa physiologie

Notebook associé au billet : https://gregs1t.github.io/trail/2026-04-01-trail-analyse-3-physiologie/

**Pré-requis** : avoir exécuté les notebooks 1 et 2 — le DataFrame `df` doit contenir
`slope_pct`, `gap_s_per_km`, `is_walk`, `speed_kmh`, `pace_s_per_km`.

**Ce qu'on fait ici :**
- Dérive cardiaque (ratio FC/vitesse lissé)
- Dégradation d'allure GAP-normalisée
- Analyse croisée pente × FC × GAP + comparaison Minetti
- TRIMP (Training Impulse, Banister 1991)
- Heatmap vitesse × pente → FC médiane

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ces variables doivent être définies (copie depuis les notebooks précédents si besoin)
# FC_MAX = 185
# FC_MIN = 47
# RAVITO_KM  = [19.2, 34.0, 45.0, 58.8, 65.4]
# RAVITO_NOM = ["St Christo", "Ste Catherine", "St Genou", "Soucieu", "Chaponost"]

SLOPE_BINS = [-30, -15, -10, -7, -5, -3, -1, 1, 3, 5, 7, 10, 15, 30]
SEXE = "H"   # 'H' ou 'F' pour le coefficient TRIMP

## 1. Dérive cardiaque

Ratio FC / vitesse lissé sur 5 min. Si ce ratio monte au fil du temps à effort constant, le cœur travaille de plus en plus — signe de fatigue, déshydratation ou montée en température.

In [ ]:
def compute_cardiac_drift(df, smoothing_sec=300):
    """Compute smoothed HR/speed ratio as a cardiac drift proxy."""
    df = df.copy()
    spd = df["speed_kmh"].replace(0, np.nan)
    df["hr_speed_ratio"] = df["heart_rate"] / spd
    df_t = df.set_index("timestamp")
    df["hr_speed_smooth"] = (
        df_t["hr_speed_ratio"]
        .rolling(f"{smoothing_sec}s", min_periods=30)
        .mean()
        .values
    )
    return df


if "heart_rate" in df.columns:
    df = compute_cardiac_drift(df)

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(df["time_h"], df["hr_speed_smooth"], color="purple", linewidth=1.5)

    if len(RAVITO_KM) > 0:
        for km, nom in zip(RAVITO_KM, RAVITO_NOM):
            idx = (df["dist_m"] / 1000.0 - km).abs().idxmin()
            t_rav = df.loc[idx, "time_h"]
            ax.axvline(t_rav, color="grey", linestyle=":", alpha=0.5, linewidth=1)
            ax.text(t_rav, ax.get_ylim()[1], nom, fontsize=7,
                    rotation=90, va="top", ha="right", color="grey")

    ax.set_xlabel("Temps (h)")
    ax.set_ylabel("FC / vitesse (lissé 5 min)")
    ax.set_title("Dérive cardiaque")
    ax.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Pas de données FC.")

## 2. Dégradation d'allure GAP-normalisée

Le GAP (allure corrigée du dénivelé) par tranche de distance montre la fatigue indépendamment du profil.
Une hausse du GAP médian au fil des km = ralentissement non expliqué par le terrain.

In [ ]:
def gap_drift(df, ravito_km, ravito_nom, n_bins=20):
    """Plot rolling median GAP to visualize pace degradation."""
    df = df.copy()
    df["dist_km"] = df["dist_m"] / 1000.0
    df["dist_bin"] = pd.cut(
        df["dist_km"],
        bins=np.linspace(0, df["dist_km"].max(), n_bins + 1)
    )
    gap_by_bin = (
        df.dropna(subset=["gap_s_per_km"])
        .groupby("dist_bin", observed=True)["gap_s_per_km"]
        .median()
    )
    bin_centers = gap_by_bin.index.map(lambda x: x.mid)

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(bin_centers, gap_by_bin.values,
            color="darkorange", linewidth=2, marker="o", markersize=4)

    for km, nom in zip(ravito_km, ravito_nom):
        ax.axvline(km, color="grey", linestyle=":", alpha=0.5, linewidth=1)
        ax.text(km, ax.get_ylim()[1], nom, fontsize=7,
                rotation=90, va="top", ha="right", color="grey")

    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("GAP médian (s/km)")
    ax.set_title("Dégradation d'allure GAP-normalisée\n"
                 "(hausse = ralentissement non expliqué par le profil)")
    ax.invert_yaxis()   # allure rapide en haut
    ax.grid(True)
    plt.tight_layout()
    plt.show()


if "gap_s_per_km" in df.columns:
    gap_drift(df, RAVITO_KM, RAVITO_NOM)
else:
    print("gap_s_per_km absent — exécute d'abord le notebook 2.")

## 3. Analyse croisée pente × FC × GAP

Médianes par bin de pente. Comparaison de la courbe GAP individuelle avec le polynôme de Minetti.

In [ ]:
def slope_crossanalysis(df, bins):
    """Aggregate GAP and HR by slope bin (median)."""
    df = df.copy()
    df["slope_bin"] = pd.cut(df["slope_pct"], bins=bins)
    cols = ["slope_pct", "gap_s_per_km"]
    if "heart_rate" in df.columns:
        cols.append("heart_rate")
    tmp = df.dropna(subset=cols)
    agg = {
        "n": ("slope_pct", "size"),
        "slope_med": ("slope_pct", "median"),
        "gap_med": ("gap_s_per_km", "median"),
    }
    if "heart_rate" in df.columns:
        agg["hr_med"] = ("heart_rate", "median")
    return tmp.groupby("slope_bin", observed=True).agg(**agg).reset_index()


summary = slope_crossanalysis(df, SLOPE_BINS)
print(summary[[c for c in ["slope_bin", "n", "slope_med", "gap_med", "hr_med"]
               if c in summary.columns]].to_string(index=False))

In [ ]:
def minetti_cost_ratio(slope_pct):
    """Cost ratio relative to flat ground from Minetti et al. (2002)."""
    i = slope_pct / 100.0
    cr = (155.4 * i**5 - 30.4 * i**4 - 43.3 * i**3
          + 46.3 * i**2 + 19.5 * i + 3.6)
    return np.clip(cr / 3.6, 0.1, None)


# Allure de référence sur le plat (médiane du GAP pour les sections < |3%|)
flat_mask = summary["slope_med"].abs() < 3
pace_flat_est = float(summary.loc[flat_mask, "gap_med"].median())

slopes_ref = np.linspace(-20, 25, 100)
minetti_pred = pace_flat_est * minetti_cost_ratio(slopes_ref)

ncols = 2 if "hr_med" in summary.columns else 1
fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5))
if ncols == 1:
    axes = [axes]

axes[0].plot(summary["slope_med"], summary["gap_med"],
             marker="o", label="Données réelles", color="darkorange")
axes[0].plot(slopes_ref, minetti_pred,
             linestyle="--", color="steelblue", label="Minetti (2002)")
axes[0].set_xlabel("Pente médiane (%)")
axes[0].set_ylabel("GAP médian (s/km)")
axes[0].set_title("Courbe GAP individuelle vs Minetti")
axes[0].legend()
axes[0].grid(True)

if "hr_med" in summary.columns:
    axes[1].plot(summary["slope_med"], summary["hr_med"],
                 marker="o", color="crimson")
    axes[1].set_xlabel("Pente médiane (%)")
    axes[1].set_ylabel("FC médiane (bpm)")
    axes[1].set_title("FC vs pente")
    axes[1].grid(True)

plt.tight_layout()
plt.show()

## 4. TRIMP (Training Impulse)

Charge d'entraînement en une valeur comparable entre des sorties de durée et d'intensité différentes.
Formule de Banister (1991) : intégrale de la FC de réserve normalisée, pondérée exponentiellement.

In [ ]:
def compute_trimp(df, fc_min, fc_max, hr_col="heart_rate", sexe="H"):
    """Compute TRIMP (Banister 1991). sexe: 'H' (b=1.92) or 'F' (b=1.67)."""
    b = 1.92 if sexe == "H" else 1.67
    tmp = df.dropna(subset=[hr_col, "timestamp"]).copy()
    tmp = tmp.sort_values("timestamp")
    dt_min = tmp["timestamp"].diff().dt.total_seconds().fillna(1.0) / 60.0
    hrr = ((tmp[hr_col] - fc_min) / (fc_max - fc_min)).clip(0, 1)
    return float((dt_min * hrr * np.exp(b * hrr)).sum())


if "heart_rate" in df.columns:
    trimp_total = compute_trimp(df, FC_MIN, FC_MAX, sexe=SEXE)
    print(f"TRIMP total : {trimp_total:.0f}")
else:
    print("Pas de données FC.")

In [ ]:
# TRIMP par section
if "heart_rate" in df.columns and len(RAVITO_KM) > 0:
    bounds = np.concatenate((
        [float(df["dist_m"].min() / 1000.0)],
        np.array(RAVITO_KM, dtype=float),
        [float(df["dist_m"].max() / 1000.0)]
    ))
    labels_sec = []
    for i in range(len(bounds) - 1):
        if i == 0:
            labels_sec.append(f"Départ → {RAVITO_NOM[0]}")
        elif i == len(bounds) - 2:
            labels_sec.append(f"{RAVITO_NOM[-1]} → Arrivée")
        else:
            labels_sec.append(f"{RAVITO_NOM[i-1]} → {RAVITO_NOM[i]}")

    df["section_id"] = np.searchsorted(
        bounds[1:], df["dist_m"].to_numpy() / 1000.0, side="right"
    )
    rows = []
    for i, lbl in enumerate(labels_sec):
        sec = df[df["section_id"] == i]
        if len(sec) > 10:
            rows.append({"Section": lbl,
                         "TRIMP": round(compute_trimp(sec, FC_MIN, FC_MAX, sexe=SEXE), 1)})
    print(pd.DataFrame(rows).to_string(index=False))

## 5. Heatmap vitesse × pente → FC médiane

Carte 2D de l'effort physiologique. Cellules avec < 5 points masquées pour éviter le bruit.

In [ ]:
def plot_heatmap_speed_slope_hr(df, fc_max):
    """2D heatmap: speed (x) × slope (y) → median HR (color)."""
    req = ["speed_kmh", "slope_pct", "heart_rate"]
    dh = df.dropna(subset=req).copy()

    speed_bins = np.arange(0, 18.5, 0.5)
    slope_bins = np.arange(-25, 26, 1.0)

    sidx = np.digitize(dh["speed_kmh"].to_numpy(), speed_bins) - 1
    pidx = np.digitize(dh["slope_pct"].to_numpy(), slope_bins) - 1

    sumhr = np.zeros((len(slope_bins), len(speed_bins)))
    count = np.zeros_like(sumhr)

    for si, pi, hr in zip(sidx, pidx, dh["heart_rate"].to_numpy()):
        if 0 <= pi < len(slope_bins) and 0 <= si < len(speed_bins):
            sumhr[pi, si] += hr
            count[pi, si] += 1

    with np.errstate(invalid="ignore"):
        heat = np.where(count > 5, sumhr / count, np.nan)

    fig, ax = plt.subplots(figsize=(11, 7))
    im = ax.imshow(
        heat, aspect="auto", origin="lower",
        extent=[0, 18, -25, 25],
        cmap="RdYlGn_r",
        vmin=fc_max * 0.55, vmax=fc_max * 0.95
    )
    plt.colorbar(im, ax=ax, label="FC médiane (bpm)")
    ax.axhline(0, color="white", linewidth=0.8, linestyle="--", alpha=0.4)
    ax.set_xlabel("Vitesse (km/h)")
    ax.set_ylabel("Pente (%)")
    ax.set_title("FC médiane par (vitesse × pente)")
    fig.tight_layout()
    plt.show()


if "heart_rate" in df.columns:
    plot_heatmap_speed_slope_hr(df, FC_MAX)
else:
    print("Pas de données FC.")

---
Fin de la série. Les notebooks 1, 2 et 3 peuvent être enchaînés sur la même session Jupyter
ou regroupés en un seul fichier pour les courses futures.